# FraudShield on REAL data (IEEE-CIS or PaySim) - Kaggle runbook
Run cells top to bottom. Two datasets are supported; **do IEEE-CIS first**, then (optionally) PaySim in a second session by changing `DATASET`.

**Before you start**
1. Settings -> **Internet On**.
2. *Add Data* -> attach your `fraudshield` code dataset (the latest zip, unzipped).
3. *Add Data* -> attach the real data:
   * **IEEE-CIS:** *Add Data -> Competition ->* `IEEE-CIS Fraud Detection` (open the competition page once and **accept the rules**, otherwise it will not attach). Files used: `train_transaction.csv`, `train_identity.csv`.
   * **PaySim:** *Add Data -> Datasets ->* search `PaySim Synthetic Financial Datasets` (slug `ealaxi/paysim1`). File used: `PS_20174392719_1491204439457_log.csv`.

**What to expect (honest):** IEEE-CIS fraud rate is ~3.5% (the task's "<1%" is true for PaySim, 0.13%). IEEE has no user id (we build a proxy), so history-based features are weaker than on the synthetic data; our 22 behavioural features are a deliberate design, not the full IEEE feature set, so expect a *moderate* PR-AUC well above the 0.035 baseline, not a leaderboard score. PaySim has no real devices/cards (derived from an account hash), so the graph part is expected to add little there - the selection logic should then fall back to the weighted blend, which is a valid finding.

In [1]:
DATASET = "ieee"        # "ieee" or "paysim"
import glob, os, shutil, yaml
def find(pattern):
    for depth in range(1, 7):
        h = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if h: return h[0]
    return None

code_params = find("params.yaml"); assert code_params, "Attach the fraudshield CODE dataset (Add Data)"
PROJ = "/kaggle/working/fraudshield"
shutil.copytree(os.path.dirname(code_params), PROJ, dirs_exist_ok=True); os.chdir(PROJ)
assert "REPORTS_DIR" in open("src/fraudshield/config.py").read(), "old code: upload the latest fraudshield.zip as a new dataset version and re-attach it"

RUN = f"/kaggle/working/run_{DATASET}"
for d in ("data", "artifacts", "reports"): os.makedirs(f"{RUN}/{d}", exist_ok=True)
if DATASET == "ieee":
    tr, idn = find("train_transaction.csv"), find("train_identity.csv")
    assert tr, "IEEE-CIS not attached: Add Data -> Competition -> IEEE-CIS Fraud Detection (accept the rules first)"
    over = {"data": {"source": "ieee", "ieee": {"transaction_csv": tr, "identity_csv": idn or ""}}, "gnn": {"epochs": 150}, "blend": {"review_cost": 20.0}}
else:
    csv = find("PS_20174392719_1491204439457_log.csv") or (glob.glob("/kaggle/input/*/PS_*.csv") or [None])[0]
    assert csv, "PaySim not attached: Add Data -> search 'PaySim Synthetic Financial Datasets'"
    over = {"data": {"source": "paysim", "paysim": {"csv": csv, "max_rows": 500000}}, "gnn": {"epochs": 60}, "blend": {"review_cost": 500.0}}
yaml.safe_dump(over, open(f"{RUN}/params.override.yaml", "w"))
os.environ.update(FRAUD_PARAMS=f"{RUN}/params.override.yaml", DATA_DIR=f"{RUN}/data", ARTIFACTS_DIR=f"{RUN}/artifacts", REPORTS_DIR=f"{RUN}/reports",
                  DATABASE_URL=f"sqlite:///{RUN}/fraudshield.db", MLFLOW_TRACKING_URI=f"sqlite:///{RUN}/mlflow.db", FRAUD_EXPERIMENT=f"fraudshield-{DATASET}")
print("dataset:", DATASET, "| output folder:", RUN); print(yaml.safe_dump(over))

dataset: ieee | output folder: /kaggle/working/run_ieee
blend:
  review_cost: 20.0
data:
  ieee:
    identity_csv: /kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
    transaction_csv: /kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
  source: ieee
gnn:
  epochs: 150



In [2]:
!pip install -q torch_geometric mlflow imbalanced-learn sqlalchemy pytest httpx fastapi evidently
!pip install -q -e . --no-deps
!pytest -q 2>&1 | tail -3

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fraudshield (pyproject.toml) ... done

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
23 passed, 149 warnings in 49.04s


## Step 2 - Real-data pipeline (IEEE ~15-25 min, PaySim ~10-15 min)
Checkpoints: `data` prints the real fraud rate (IEEE ~3.5%, PaySim ~0.13%); XGBoost prints `test_pr_auc` (compare with `test_prevalence`); `blend` prints which method was chosen.

In [3]:
!python -m fraudshield.data
!python -m fraudshield.features
!python scripts/eda.py

590,540 txns | fraud rate 3.499% | users 90,375
{'train': 354324, 'val': 118108, 'test': 118108} | cutoff 2018-04-21 02:54:02.472431+00:00
{
  "n_txn": 590540,
  "n_users": 90375,
  "fraud_rate": 0.03499000914417313,
  "imbalance_ratio": 27.579586700866283,
  "fraud_by_type": {
    "CASH_IN": 0.04766,
    "CASH_OUT": 0.03783,
    "DEBIT": 0.059,
    "PAYMENT": 0.0204,
    "TRANSFER": 0.11687
  },
  "median_amount_legit": 68.5,
  "median_amount_fraud": 75.0,
  "fraud_device_change_rate": 0.16657794124764072,
  "legit_device_change_rate": 0.07271393651612541,
  "fraud_geo_change_rate": 9.679136621013405e-05,
  "legit_geo_change_rate": 0.0001561740515935895
}


In [4]:
!python -m fraudshield.train_xgb
!python -m fraudshield.graph
!python -m fraudshield.blend

2026/11/16 12:08:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/16 12:08:14 INFO mlflow.store.db.utils: Updating database tables
2026/11/16 12:08:19 INFO mlflow.tracking.fluent: Experiment with name 'fraudshield-ieee' does not exist. Creating a new experiment.
{
  "val_pr_auc": 0.1680289406285464,
  "val_roc_auc": 0.7820873196615504,
  "val_prevalence": 0.03904053916754157,
  "val_precision_at_recall_80": 0.07477450086145739,
  "val_precision_at_recall_50": 0.14501320588605207,
  "test_pr_auc": 0.13422849484583416,
  "test_roc_auc": 0.7629295256033933,
  "test_prevalence": 0.034409184813899145,
  "test_precision_at_recall_80": 0.0599071549627883,
  "test_precision_at_recall_50": 0.11594616174289951
}
{'val_ap': 0.309322223752806, 'test_ap': 0.3427956669339097, 'test_auc': 0.7976266890689296, 'n_users': 70385, 'n_fraud_users': 3468}
{
  "selection_on_validation": {
    "xgb_ap": 0.1680289406285464,
    "weighted_ap": 0.16807716089469202,
    "stack_oof

In [5]:
!python -m fraudshield.replay_local --limit 20000
!python -m fraudshield.monitoring
!python -m fraudshield.ablation

{
  "n": 20000,
  "fraud_rate": 0.0339,
  "online_pr_auc": 0.1270368748168311,
  "alert_precision": 0.18717948717948718,
  "fraud_recall": 0.10766961651917405,
  "alerts": 390,
  "p50_latency_ms": 3.7032550000049014,
  "p99_latency_ms": 7.033976579955362,
  "max_abs_feature_diff_online_vs_offline": 0.0
}
/usr/local/lib/python3.13/dist-packages/evidently/core/datasets.py:1760: RuntimeWarning: invalid value encountered in scalar divide
  infinite=StatCountValue(infinite_count, infinite_count / data.count()),
{
  "generated_at": "2026-11-16T12:16:10.025871+00:00",
  "n_reference": 20000,
  "n_current": 5000,
  "share_drifted": 0.1,
  "dataset_drift": false,
  "evidently_html": true,
  "retrain_triggered": false,
  "retrain_reasons": [],
  "status": "ok"
}
                                      val_oof_ap  test_ap  precision  recall  ring_fraud_recall  net_savings
xgb only                                   0.167    0.134      0.227   0.117                NaN    17514.379
xgb + cluster size 

## Step 3 - Results (screenshot this cell; these are the numbers for your slides)

In [6]:
import json, os, pandas as pd, sqlite3
A, R = os.environ["ARTIFACTS_DIR"], os.environ["REPORTS_DIR"]
print("EDA:", json.dumps({k: v for k, v in json.load(open(f"{R}/eda.json")).items() if k in ("n_txn", "n_users", "fraud_rate", "imbalance_ratio", "fraud_by_type")}, indent=1))
m = json.load(open(f"{A}/meta.json"))["metrics"]
print("selection:", json.dumps(m["selection_on_validation"], indent=1))
rows = []
for k in ("xgb", "weighted", "stack"):
    t = m["test"][k]; b = t["at_threshold"]
    rows.append(dict(method=k, pr_auc=t["pr_auc"], baseline=t["prevalence"], lift=t["pr_auc"] / t["prevalence"], roc_auc=t["roc_auc"],
                     p_at_50_recall=t["precision_at_recall_50"], precision=b["precision"], recall=b["recall"], alerts_per_1k=b["alerts_per_1k_txn"], net_savings=b["net_savings"]))
display(pd.DataFrame(rows).round(3)); print("chosen method:", m["chosen_method"])
print("bootstrap PR-AUC gain vs XGBoost:", {k: {a: round(b, 3) for a, b in v.items()} for k, v in m["test"]["bootstrap_ap_gain_vs_xgb"].items()})
print("GNN alone on covered txns:", m["test"]["gnn_only_on_covered_txns"], "| graph coverage:", m["test"]["graph_coverage"])
print("online replay:", json.load(open(f"{A}/metrics_online_replay_memory.json")))
ab = json.load(open(f"{A}/ablation.json")); boot = ab.pop("_bootstrap_ap_gain"); display(pd.DataFrame(ab).T[["val_oof_ap", "test_ap", "precision", "recall", "net_savings"]].round(3))
print("ablation bootstrap:", {k: {a: round(b, 3) for a, b in v.items() if a in ("mean_gain", "ci_low", "ci_high")} for k, v in boot.items()})
d = json.load(open(f"{R}/drift_latest.json")); print("drift:", {k: d.get(k) for k in ("share_drifted", "dataset_drift", "retrain_triggered", "evidently_html")})

EDA: {
 "n_txn": 590540,
 "n_users": 90375,
 "fraud_rate": 0.03499000914417313,
 "imbalance_ratio": 27.579586700866283,
 "fraud_by_type": {
  "CASH_IN": 0.04766,
  "CASH_OUT": 0.03783,
  "DEBIT": 0.059,
  "PAYMENT": 0.0204,
  "TRANSFER": 0.11687
 }
}
selection: {
 "xgb_ap": 0.1680289406285464,
 "weighted_ap": 0.16807716089469202,
 "stack_oof_ap": 0.16757477721322564,
 "stack_C": 0.1,
 "min_gain": 0.01,
 "chosen": "weighted",
 "alpha_xgb_weight": 0.95
}


,method,pr_auc,baseline,lift,roc_auc,p_at_50_recall,precision,recall,alerts_per_1k,net_savings
0,xgb,0.134,0.034,3.901,0.763,0.116,0.228,0.119,18.009,17340.337
1,weighted,0.136,0.034,3.939,0.763,0.118,0.224,0.137,20.981,13130.823
2,stack,0.138,0.034,4.001,0.769,0.123,0.270,0.071,9.110,16342.971


chosen method: weighted
bootstrap PR-AUC gain vs XGBoost: {'weighted': {'mean_gain': 0.001, 'ci_low': -0.0, 'ci_high': 0.003, 'prob_better': 0.965, 'n_boot': 200}, 'stack': {'mean_gain': 0.004, 'ci_low': -0.0, 'ci_high': 0.007, 'prob_better': 0.96, 'n_boot': 200}}
GNN alone on covered txns: {'pr_auc': 0.06948490317673343, 'roc_auc': 0.6477448409271576, 'prevalence': 0.034363168407349626, 'precision_at_recall_80': 0.04049326068253513, 'precision_at_recall_50': 0.05887454551519397} | graph coverage: 0.8520252650116842
online replay: {'n': 20000, 'fraud_rate': 0.0339, 'online_pr_auc': 0.1270368748168311, 'alert_precision': 0.18717948717948718, 'fraud_recall': 0.10766961651917405, 'alerts': 390, 'p50_latency_ms': 3.7032550000049014, 'p99_latency_ms': 7.033976579955362, 'max_abs_feature_diff_online_vs_offline': 0.0}


,val_oof_ap,test_ap,precision,recall,net_savings
xgb only,0.167,0.134,0.227,0.117,17514.379
xgb + cluster size,0.167,0.135,0.218,0.146,20405.527
xgb + GNN score,0.168,0.137,0.208,0.171,12772.734
xgb + GNN + cluster (no interaction),0.168,0.138,0.268,0.070,15090.597
full stack,0.168,0.138,0.270,0.071,16342.971


ablation bootstrap: {'cluster_vs_xgb': {'mean_gain': 0.001, 'ci_low': 0.0, 'ci_high': 0.001}, 'full_vs_cluster': {'mean_gain': 0.003, 'ci_low': -0.001, 'ci_high': 0.007}, 'gnn_vs_cluster': {'mean_gain': 0.002, 'ci_low': -0.002, 'ci_high': 0.006}}
drift: {'share_drifted': 0.1, 'dataset_drift': False, 'retrain_triggered': False, 'evidently_html': True}


In [7]:
shutil.make_archive(f"/kaggle/working/fraudshield_results_{DATASET}", "zip", RUN)
print("download: /kaggle/working/fraudshield_results_%s.zip  (then Save Version -> Save & Run All)" % DATASET)

download: /kaggle/working/fraudshield_results_ieee.zip  (then Save Version -> Save & Run All)


### Notes
* Outputs live in `/kaggle/working/run_<dataset>/` (data, artifacts, reports, databases), so IEEE and PaySim runs never overwrite each other or your synthetic results.
* Run a dataset in a **fresh session** (restart kernel) when you change `DATASET`.
* If a step fails, copy the **last 15 lines** of the error and send them; the loaders were validated on small mock files with the real column names but have not been run on the full real files.
* Do **not** compare these numbers with the synthetic-data numbers: different data, different difficulty. Report them separately and state the limits (IEEE: user proxy; PaySim: synthetic devices/cards).